# Evaluating Language Models: Metrics, Methods, and Best Practices

## Introduction: Why Evaluation Matters More Than You Think

There is a famous saying in machine learning that you cannot improve what you cannot measure. This principle becomes especially critical when working with large language models, where the space of possible outputs is vast and the quality of generation can vary dramatically based on subtle changes in the model or prompt. Without rigorous evaluation, you are essentially flying blind, unable to tell whether your changes are making things better or worse.

Consider this scenario. You have built a language model for customer service responses. The model generates fluent, natural-sounding text. Your stakeholders are impressed by the examples you show them. But how do you know the model will perform well on the full range of customer queries? How do you measure whether version two is actually better than version one? How do you catch cases where the model produces offensive or inappropriate responses before they reach real users?

These questions all require systematic evaluation. Simply generating a few examples and eyeballing the results is not enough for production systems. You need quantitative metrics that you can track over time, qualitative assessment procedures that capture aspects metrics miss, and robust testing frameworks that reveal edge cases and failure modes.

### What This Notebook Covers

Through this comprehensive guide, you will learn how to evaluate language models across multiple dimensions. We will start with automatic metrics that can be computed efficiently without human input. These include perplexity for measuring how well a model predicts text, BLEU and ROUGE for comparing generated text to references, and embedding-based metrics that capture semantic similarity. Each metric has strengths and limitations, and understanding these trade-offs is crucial for choosing the right evaluation approach.

We will then explore task-specific evaluation, showing how to measure performance on downstream applications like question answering, summarization, and classification. Different tasks require different evaluation strategies, and we will implement practical frameworks for each.

Next, we will tackle the challenge of human evaluation. While automatic metrics are convenient, they often fail to capture important aspects of quality like factual accuracy, coherence, and appropriateness. We will design human evaluation protocols that produce reliable, actionable insights while being feasible to run in practice.

Finally, we will address critical concerns around bias, fairness, and safety. Language models can perpetuate harmful stereotypes, generate toxic content, or fail catastrophically on certain inputs. We will implement techniques for detecting and measuring these issues before they cause real-world harm.

By the end of this notebook, you will have a comprehensive toolkit for evaluating language models rigorously and thoughtfully. You will understand not just how to compute various metrics, but when each metric is appropriate, what it actually measures, and how to interpret the results. This knowledge is essential for anyone building production language model systems or conducting research that aims to advance the state of the art.

Let us begin by setting up our environment and understanding the foundational concepts that underpin all evaluation approaches.

## Part 1: Environment Setup and Core Concepts

Before diving into specific metrics, we need to establish our computational environment and understand the fundamental principles of language model evaluation. The libraries we will use represent industry standards for evaluation work, each chosen for specific capabilities that make our evaluation tasks more efficient and reliable.

Evaluation is fundamentally about comparison. We compare generated text to reference text, compare model predictions to gold labels, or compare different model versions to each other. The challenge lies in defining what makes one output better than another. Is fluency more important than accuracy? Should we prioritize brevity or completeness? These questions have no universal answers, which is why we need multiple evaluation approaches that capture different aspects of quality.

Another key concept is the distinction between intrinsic and extrinsic evaluation. Intrinsic metrics measure properties of the model or its outputs directly, such as perplexity or fluency scores. Extrinsic metrics measure performance on downstream tasks that users care about, such as question answering accuracy or customer satisfaction. Both types of evaluation are valuable, and the best evaluation strategies combine them to get a complete picture of model quality.

In [1]:
# Install required packages
# Uncomment the following line if you need to install packages
# !pip install transformers torch datasets nltk rouge-score bert-score sacrebleu sentencepiece perspective

import torch
import numpy as np
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    AutoModelForSeq2SeqLM
)
from datasets import load_dataset
import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer
from bert_score import score as bert_score
import matplotlib.pyplot as plt
import seaborn as sns
from typing import List, Dict, Tuple, Optional
from dataclasses import dataclass
import pandas as pd
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

# Download NLTK data
nltk.download('punkt', quiet=True)
nltk.download('wordnet', quiet=True)

# Set random seeds
torch.manual_seed(42)
np.random.seed(42)

# Configure plotting
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Determine device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

print("\nEnvironment setup complete!")
print("\nCore evaluation concepts:")
print("  • Automatic metrics: Fast, scalable, but may miss important quality aspects")
print("  • Human evaluation: Captures nuanced quality, but expensive and time-consuming")
print("  • Task-specific metrics: Measure performance on actual use cases")
print("  • Safety evaluation: Detect bias, toxicity, and harmful outputs")
print("\nNo single metric tells the complete story. Use multiple evaluation approaches.")

Using device: cpu

Environment setup complete!

Core evaluation concepts:
  • Automatic metrics: Fast, scalable, but may miss important quality aspects
  • Human evaluation: Captures nuanced quality, but expensive and time-consuming
  • Task-specific metrics: Measure performance on actual use cases
  • Safety evaluation: Detect bias, toxicity, and harmful outputs

No single metric tells the complete story. Use multiple evaluation approaches.


## Part 2: Perplexity - The Foundation Metric

Perplexity is perhaps the most fundamental metric for evaluating language models. It measures how surprised the model is by a piece of text. Formally, perplexity is the exponentiated average negative log-likelihood of the tokens in the sequence. When a model assigns high probability to the actual tokens that appear, perplexity is low. When the model is surprised by the tokens, perplexity is high.

Think of perplexity as measuring how many equally likely choices the model thinks it has at each position. A perplexity of one means the model perfectly predicts every token. A perplexity of one hundred means the model is as uncertain as if it had to choose uniformly among one hundred options at each step. Lower perplexity generally indicates a better language model, though this metric has important limitations we will discuss.

Perplexity is useful because it is computed efficiently on any text without requiring human annotations. It also correlates reasonably well with model quality for many tasks. However, perplexity only measures how well the model predicts token sequences from its training distribution. It does not directly measure generation quality, factual accuracy, or usefulness for downstream tasks. A model can have low perplexity while still generating nonsensical or harmful text.

In [2]:
class PerplexityEvaluator:
    """Compute perplexity for evaluating language models.
    
    Perplexity measures how well the model predicts a sample of text. It is computed
    as the exponentiated average negative log-likelihood of the tokens. Lower values
    indicate better prediction quality.
    
    The formula is: PPL = exp(-1/N * Σ log P(token_i | context))
    where N is the number of tokens.
    """
    
    def __init__(self, model_name: str = 'gpt2'):
        """Initialize with a language model.
        
        Args:
            model_name: Name or path of the model to evaluate
        """
        print(f"Loading model: {model_name}")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForCausalLM.from_pretrained(model_name)
        self.model.to(device)
        self.model.eval()
        print("Model loaded successfully")
    
    def compute_perplexity(self, text: str) -> float:
        """Compute perplexity for a piece of text.
        
        This method encodes the text, computes the loss (cross-entropy), and
        exponentiates it to get perplexity. The computation is done without
        gradients since we are only evaluating, not training.
        """
        # Tokenize the text
        encodings = self.tokenizer(text, return_tensors='pt').to(device)
        
        # Compute loss
        with torch.no_grad():
            outputs = self.model(**encodings, labels=encodings['input_ids'])
            loss = outputs.loss
        
        # Perplexity is exp(loss)
        perplexity = torch.exp(loss).item()
        return perplexity
    
    def compute_perplexity_batch(self, texts: List[str]) -> List[float]:
        """Compute perplexity for multiple texts efficiently.
        
        Processing texts in batches is more efficient than one at a time,
        but we need to handle different text lengths carefully.
        """
        perplexities = []
        
        for text in texts:
            ppl = self.compute_perplexity(text)
            perplexities.append(ppl)
        
        return perplexities
    
    def compare_texts(self, texts: List[str], labels: List[str]) -> pd.DataFrame:
        """Compare perplexities across different texts.
        
        This is useful for understanding which types of text the model
        handles well versus poorly. Lower perplexity indicates the model
        finds that type of text more predictable.
        """
        perplexities = self.compute_perplexity_batch(texts)
        
        results = pd.DataFrame({
            'text': [t[:50] + '...' if len(t) > 50 else t for t in texts],
            'label': labels,
            'perplexity': perplexities
        })
        
        return results.sort_values('perplexity')


# Initialize evaluator
ppl_evaluator = PerplexityEvaluator('gpt2')

# Test with different types of text
test_texts = [
    "The quick brown fox jumps over the lazy dog.",  # Common English
    "Machine learning models learn patterns from data.",  # Technical but clear
    "Quantum entanglement exhibits non-local correlations.",  # Technical jargon
    "Buffalo buffalo Buffalo buffalo buffalo buffalo Buffalo buffalo.",  # Grammatical but confusing
    "asdf jkl; qwer uiop zxcv bnm,",  # Random characters
]

labels = ['Common English', 'Clear Technical', 'Dense Technical', 'Confusing', 'Random']

print("\nComputing perplexity for different text types...\n")
results = ppl_evaluator.compare_texts(test_texts, labels)
print(results.to_string(index=False))

print("\n" + "="*80)
print("INTERPRETING PERPLEXITY")
print("="*80)
print("""
Lower perplexity means the model finds the text more predictable:
  • Common English has low perplexity because the model has seen similar text often
  • Technical text may have higher perplexity if outside the training distribution
  • Random characters have very high perplexity - the model cannot predict them

Important caveats:
  • Perplexity measures predictability, not quality or usefulness
  • A model can have low perplexity on harmful or incorrect text
  • Perplexity is domain-dependent - compare only within similar domains
  • Models trained on different data will have different perplexity baselines
""")

Loading model: gpt2
Model loaded successfully

Computing perplexity for different text types...



`loss_type=None` was set in the config but it is unrecognised.Using the default loss: `ForCausalLMLoss`.


                                                 text           label  perplexity
Quantum entanglement exhibits non-local correlatio... Dense Technical   63.934849
Buffalo buffalo Buffalo buffalo buffalo buffalo Bu...       Confusing   67.723236
    Machine learning models learn patterns from data. Clear Technical  148.699417
         The quick brown fox jumps over the lazy dog.  Common English  162.470215
                        asdf jkl; qwer uiop zxcv bnm,          Random 2006.819092

INTERPRETING PERPLEXITY

Lower perplexity means the model finds the text more predictable:
  • Common English has low perplexity because the model has seen similar text often
  • Technical text may have higher perplexity if outside the training distribution
  • Random characters have very high perplexity - the model cannot predict them

Important caveats:
  • Perplexity measures predictability, not quality or usefulness
  • A model can have low perplexity on harmful or incorrect text
  • Perplexity is 

## Part 3: BLEU Score - Measuring Translation and Generation Quality

BLEU, which stands for Bilingual Evaluation Understudy, was originally developed for machine translation but has become widely used for evaluating any text generation task. The core idea is simple yet powerful: compare the generated text to one or more reference texts and measure how many word sequences they have in common.

BLEU examines n-grams of different lengths (typically from one to four words) and counts how many of these n-grams appear in both the generated text and the references. It also includes a brevity penalty to discourage generating very short outputs that trivially match the reference by simply omitting content. The score ranges from zero to one, with higher scores indicating greater similarity to the references.

While BLEU is computationally efficient and has been extensively used in research, it has notable limitations. It only measures surface-level similarity and cannot assess semantic equivalence. Two sentences with identical meaning but different words would receive a low BLEU score. It also treats all n-gram matches equally, regardless of whether they capture important content or trivial function words. Despite these limitations, BLEU remains valuable as part of a comprehensive evaluation strategy, particularly when you have good reference texts and care about precise wording.

In [3]:
class BLEUEvaluator:
    """Compute BLEU scores for comparing generated text to references.
    
    BLEU (Bilingual Evaluation Understudy) measures n-gram overlap between
    generated text and reference texts. It is widely used for machine translation
    and other generation tasks. Scores range from 0 to 1, with higher being better.
    """
    
    def __init__(self):
        """Initialize BLEU evaluator with smoothing function.
        
        The smoothing function helps handle cases where some n-grams have zero
        matches, which would otherwise make the geometric mean undefined.
        """
        self.smoothing = SmoothingFunction().method1
    
    def compute_bleu(self, candidate: str, references: List[str], 
                     max_n: int = 4) -> Dict[str, float]:
        """Compute BLEU score comparing candidate to references.
        
        Args:
            candidate: Generated text to evaluate
            references: List of reference texts to compare against
            max_n: Maximum n-gram length to consider (typically 4)
            
        Returns:
            Dictionary with overall BLEU and individual n-gram precisions
        """
        # Tokenize texts
        candidate_tokens = nltk.word_tokenize(candidate.lower())
        reference_tokens = [nltk.word_tokenize(ref.lower()) for ref in references]
        
        # Compute BLEU with different n-gram weights
        weights = [1.0/max_n] * max_n  # Equal weights for all n-grams
        
        bleu_score = sentence_bleu(
            reference_tokens,
            candidate_tokens,
            weights=weights,
            smoothing_function=self.smoothing
        )
        
        # Also compute individual n-gram precisions
        precisions = {}
        for n in range(1, max_n + 1):
            weights_n = [1.0 if i == n-1 else 0.0 for i in range(max_n)]
            precisions[f'bleu-{n}'] = sentence_bleu(
                reference_tokens,
                candidate_tokens,
                weights=weights_n,
                smoothing_function=self.smoothing
            )
        
        return {
            'bleu': bleu_score,
            **precisions
        }
    
    def evaluate_candidates(self, candidates: List[str], 
                          references: List[List[str]]) -> pd.DataFrame:
        """Evaluate multiple candidates against their references.
        
        This is useful for comparing different models or generation strategies.
        Each candidate can have multiple references to compare against.
        """
        results = []
        
        for i, (candidate, refs) in enumerate(zip(candidates, references)):
            scores = self.compute_bleu(candidate, refs)
            results.append({
                'candidate_id': i,
                'candidate': candidate[:60] + '...' if len(candidate) > 60 else candidate,
                **scores
            })
        
        return pd.DataFrame(results)


# Initialize BLEU evaluator
bleu_evaluator = BLEUEvaluator()

print("Testing BLEU evaluation...\n")
print("="*80)

# Example 1: Perfect match
candidate1 = "The cat sat on the mat."
reference1 = ["The cat sat on the mat."]
scores1 = bleu_evaluator.compute_bleu(candidate1, reference1)
print("Example 1: Perfect match")
print(f"Candidate: {candidate1}")
print(f"Reference: {reference1[0]}")
print(f"BLEU Score: {scores1['bleu']:.4f}")
print()

# Example 2: Semantic equivalence but different words
candidate2 = "The feline rested on the rug."
reference2 = ["The cat sat on the mat."]
scores2 = bleu_evaluator.compute_bleu(candidate2, reference2)
print("Example 2: Same meaning, different words")
print(f"Candidate: {candidate2}")
print(f"Reference: {reference2[0]}")
print(f"BLEU Score: {scores2['bleu']:.4f}")
print("Note: Low score despite semantic similarity - BLEU limitation!")
print()

# Example 3: Partial match
candidate3 = "The cat sat on a comfortable mat."
reference3 = ["The cat sat on the mat."]
scores3 = bleu_evaluator.compute_bleu(candidate3, reference3)
print("Example 3: Partial match with extra content")
print(f"Candidate: {candidate3}")
print(f"Reference: {reference3[0]}")
print(f"BLEU Score: {scores3['bleu']:.4f}")
print()

# Example 4: Multiple references
candidate4 = "The cat was sitting on the mat."
references4 = [
    "The cat sat on the mat.",
    "A cat was on the mat.",
    "The cat rested on a mat."
]
scores4 = bleu_evaluator.compute_bleu(candidate4, references4)
print("Example 4: Multiple references increase score")
print(f"Candidate: {candidate4}")
print(f"References: {len(references4)} different versions")
print(f"BLEU Score: {scores4['bleu']:.4f}")
print("Note: Multiple references help capture acceptable variation")

print("\n" + "="*80)
print("BLEU SCORE INTERPRETATION")
print("="*80)
print("""
BLEU Score Ranges (rough guidelines):
  • 0.00 - 0.10: Almost no overlap, very poor quality
  • 0.10 - 0.30: Low quality, significant differences from reference
  • 0.30 - 0.50: Moderate quality, understandable but imperfect
  • 0.50 - 0.70: Good quality, captures most of the reference content
  • 0.70 - 1.00: Very high quality, very close to reference

Key Strengths:
  • Fast to compute
  • Language-independent
  • Well-established and widely used
  • Works with multiple references

Key Limitations:
  • Only measures surface-level word overlap
  • Cannot assess semantic equivalence
  • Sensitive to word choice even when meaning is preserved
  • Does not consider word order beyond n-grams
  • Requires good reference translations
""")

Testing BLEU evaluation...



LookupError: 
**********************************************************************
  Resource [93mpunkt_tab[0m not found.
  Please use the NLTK Downloader to obtain the resource:

  [31m>>> import nltk
  >>> nltk.download('punkt_tab')
  [0m
  For more information see: https://www.nltk.org/data.html

  Attempted to load [93mtokenizers/punkt_tab/english/[0m

  Searched in:
    - '/Users/yenokhakobyan/nltk_data'
    - '/Users/yenokhakobyan/miniconda3/nltk_data'
    - '/Users/yenokhakobyan/miniconda3/share/nltk_data'
    - '/Users/yenokhakobyan/miniconda3/lib/nltk_data'
    - '/usr/share/nltk_data'
    - '/usr/local/share/nltk_data'
    - '/usr/lib/nltk_data'
    - '/usr/local/lib/nltk_data'
**********************************************************************


## Part 4: ROUGE Scores - Evaluating Summarization

ROUGE, which stands for Recall-Oriented Understudy for Gisting Evaluation, was specifically designed for evaluating summaries. While BLEU emphasizes precision (what fraction of the generated words appear in the reference), ROUGE emphasizes recall (what fraction of the reference words appear in the generation). This difference reflects the priorities of summarization, where we care more about covering the important content from the source than about avoiding extra details.

There are several ROUGE variants, each measuring different aspects of summary quality. ROUGE-N measures n-gram overlap, similar to BLEU but with a recall focus. ROUGE-L measures the longest common subsequence between the texts, capturing sentence-level structure. ROUGE-W weights longer common subsequences more heavily than shorter ones. Each variant provides different insights into summary quality.

ROUGE has become the standard metric for summarization evaluation in research, though it shares some of BLEU's limitations regarding semantic understanding. It works best when you have abstractive summaries to compare against, where direct word overlap with the source is limited. For extractive summarization, high ROUGE scores are easier to achieve but may not reflect true understanding or useful summarization.

In [4]:
class ROUGEEvaluator:
    """Compute ROUGE scores for evaluating summaries.
    
    ROUGE (Recall-Oriented Understudy for Gisting Evaluation) is the standard
    metric for automatic summarization evaluation. It measures overlap between
    generated and reference summaries, with an emphasis on recall.
    """
    
    def __init__(self, rouge_types: Optional[List[str]] = None):
        """Initialize ROUGE evaluator.
        
        Args:
            rouge_types: List of ROUGE variants to compute.
                        Common options: 'rouge1', 'rouge2', 'rougeL'
        """
        if rouge_types is None:
            rouge_types = ['rouge1', 'rouge2', 'rougeL']
        
        self.scorer = rouge_scorer.RougeScorer(rouge_types, use_stemmer=True)
        self.rouge_types = rouge_types
    
    def compute_rouge(self, candidate: str, reference: str) -> Dict[str, Dict[str, float]]:
        """Compute ROUGE scores for a candidate summary.
        
        Returns precision, recall, and F1 for each ROUGE variant.
        
        Args:
            candidate: Generated summary
            reference: Reference summary
            
        Returns:
            Dictionary mapping ROUGE types to precision/recall/f1 scores
        """
        scores = self.scorer.score(reference, candidate)
        
        # Convert to more readable format
        formatted_scores = {}
        for rouge_type in self.rouge_types:
            score = scores[rouge_type]
            formatted_scores[rouge_type] = {
                'precision': score.precision,
                'recall': score.recall,
                'f1': score.fmeasure
            }
        
        return formatted_scores
    
    def evaluate_summaries(self, candidates: List[str], 
                          references: List[str]) -> pd.DataFrame:
        """Evaluate multiple summaries.
        
        This is useful for comparing different summarization systems or
        analyzing performance across a test set.
        """
        results = []
        
        for i, (candidate, reference) in enumerate(zip(candidates, references)):
            scores = self.compute_rouge(candidate, reference)
            
            result = {
                'id': i,
                'candidate': candidate[:50] + '...' if len(candidate) > 50 else candidate,
            }
            
            # Flatten scores
            for rouge_type, metrics in scores.items():
                for metric_name, value in metrics.items():
                    result[f'{rouge_type}_{metric_name}'] = value
            
            results.append(result)
        
        return pd.DataFrame(results)


# Initialize ROUGE evaluator
rouge_evaluator = ROUGEEvaluator()

print("Testing ROUGE evaluation for summarization...\n")
print("="*80)

# Sample document to summarize
document = """Artificial intelligence has made remarkable progress in recent years, 
particularly in the field of natural language processing. Large language models 
trained on vast amounts of text data have demonstrated impressive capabilities in 
understanding and generating human-like text. These models use transformer 
architectures with attention mechanisms that allow them to capture long-range 
dependencies in text. Applications include machine translation, question answering, 
summarization, and creative writing. However, challenges remain regarding bias, 
factual accuracy, and the computational resources required for training and deployment."""

# Different quality summaries
summaries = [
    # Good summary
    "AI, especially in NLP, has made significant progress with large language models using transformers. Applications include translation and summarization, but challenges like bias and computational costs persist.",
    
    # Extractive but accurate
    "Large language models trained on vast amounts of text data have demonstrated impressive capabilities. Applications include machine translation, question answering, summarization, and creative writing.",
    
    # Too brief
    "AI and NLP have improved with language models.",
    
    # Good but different phrasing
    "Recent advances in artificial intelligence have led to powerful language models that excel at various NLP tasks. These systems face ongoing challenges related to bias and resource requirements."
]

# Reference summary
reference = """Recent AI advances, particularly in NLP with transformer-based language models, 
have enabled impressive text understanding and generation capabilities across applications 
like translation and summarization, though bias and computational costs remain challenges."""

# Evaluate each summary
for i, summary in enumerate(summaries, 1):
    print(f"Summary {i}:")
    print(f"  {summary}")
    print()
    
    scores = rouge_evaluator.compute_rouge(summary, reference)
    
    print("  ROUGE Scores:")
    for rouge_type, metrics in scores.items():
        print(f"    {rouge_type.upper()}:")
        print(f"      Precision: {metrics['precision']:.4f}")
        print(f"      Recall:    {metrics['recall']:.4f}")
        print(f"      F1:        {metrics['f1']:.4f}")
    print("\n" + "-"*80 + "\n")

print("="*80)
print("ROUGE SCORE INTERPRETATION")
print("="*80)
print("""
ROUGE Variants:
  • ROUGE-1: Unigram overlap (individual word matches)
  • ROUGE-2: Bigram overlap (two-word phrase matches)
  • ROUGE-L: Longest common subsequence (sentence structure)

Metrics Explained:
  • Precision: What fraction of generated words appear in reference?
  • Recall: What fraction of reference words appear in generation?
  • F1: Harmonic mean of precision and recall

For summarization, recall is often more important than precision because
we want to ensure the summary covers the key content, even if it includes
some additional details.

Good Summary Characteristics:
  • High ROUGE-1: Covers important vocabulary from reference
  • High ROUGE-2: Captures key phrases accurately
  • High ROUGE-L: Maintains sentence structure and flow

Limitations:
  • Cannot judge semantic equivalence with different wording
  • May reward extractive summaries over abstractive ones
  • Does not measure factual accuracy or coherence
  • Sensitive to reference quality
""")

Testing ROUGE evaluation for summarization...

Summary 1:
  AI, especially in NLP, has made significant progress with large language models using transformers. Applications include translation and summarization, but challenges like bias and computational costs persist.

  ROUGE Scores:
    ROUGE1:
      Precision: 0.6296
      Recall:    0.5312
      F1:        0.5763
    ROUGE2:
      Precision: 0.2692
      Recall:    0.2258
      F1:        0.2456
    ROUGEL:
      Precision: 0.5185
      Recall:    0.4375
      F1:        0.4746

--------------------------------------------------------------------------------

Summary 2:
  Large language models trained on vast amounts of text data have demonstrated impressive capabilities. Applications include machine translation, question answering, summarization, and creative writing.

  ROUGE Scores:
    ROUGE1:
      Precision: 0.4167
      Recall:    0.3125
      F1:        0.3571
    ROUGE2:
      Precision: 0.0435
      Recall:    0.0323
   

## Part 5: BERTScore - Semantic Similarity with Embeddings

BERTScore represents a significant advance over traditional metrics like BLEU and ROUGE because it leverages contextual embeddings from pre-trained language models. Instead of simply counting word matches, BERTScore computes the similarity between the contextualized embeddings of tokens in the candidate and reference texts. This allows it to recognize semantic equivalence even when different words are used.

The key insight behind BERTScore is that modern language models like BERT learn rich representations where semantically similar words and phrases cluster together in embedding space. By comparing these embeddings rather than surface forms, we can capture meaning more accurately. For example, "automobile" and "car" would have very similar embeddings and thus receive a high BERTScore match, while BLEU would treat them as completely different.

BERTScore computes precision, recall, and F1 scores similar to ROUGE, but at the embedding level. It matches each token in the candidate to its most similar token in the reference based on cosine similarity of their embeddings. This matching process can be weighted by inverse document frequency to emphasize content words over function words, making the metric more focused on important semantic content.

While BERTScore correlates better with human judgments than traditional metrics, it is more computationally expensive and requires loading a large pre-trained model. It also inherits whatever biases are present in its underlying embedding model. Nevertheless, BERTScore has become increasingly popular for evaluation because it captures semantic quality in a way that traditional metrics cannot.

In [5]:
class BERTScoreEvaluator:
    """Compute BERTScore using contextual embeddings for semantic similarity.
    
    BERTScore leverages pre-trained language models to compute similarity between
    generated and reference texts at the embedding level. This captures semantic
    equivalence better than surface-level metrics like BLEU.
    """
    
    def __init__(self, model_type: str = 'bert-base-uncased'):
        """Initialize BERTScore evaluator.
        
        Args:
            model_type: Pre-trained model to use for embeddings
                       (e.g., 'bert-base-uncased', 'roberta-large')
        """
        self.model_type = model_type
        print(f"BERTScore will use model: {model_type}")
        print("Note: First run will download the model, which may take time.")
    
    def compute_bertscore(self, candidates: List[str], 
                         references: List[str]) -> Dict[str, List[float]]:
        """Compute BERTScore for candidates against references.
        
        Args:
            candidates: List of generated texts
            references: List of reference texts (one per candidate)
            
        Returns:
            Dictionary with precision, recall, and F1 scores
        """
        # Compute BERTScore
        P, R, F1 = bert_score(
            candidates,
            references,
            model_type=self.model_type,
            verbose=False
        )
        
        return {
            'precision': P.tolist(),
            'recall': R.tolist(),
            'f1': F1.tolist()
        }
    
    def compare_with_traditional(self, candidate: str, reference: str) -> pd.DataFrame:
        """Compare BERTScore with traditional metrics on the same example.
        
        This helps illustrate how BERTScore captures semantic similarity
        that surface-level metrics miss.
        """
        # Compute BERTScore
        bert_scores = self.compute_bertscore([candidate], [reference])
        
        # Compute BLEU
        bleu_eval = BLEUEvaluator()
        bleu_scores = bleu_eval.compute_bleu(candidate, [reference])
        
        # Compute ROUGE
        rouge_eval = ROUGEEvaluator()
        rouge_scores = rouge_eval.compute_rouge(candidate, reference)
        
        # Format results
        results = {
            'Metric': ['BLEU', 'ROUGE-1 F1', 'ROUGE-2 F1', 'ROUGE-L F1', 'BERTScore F1'],
            'Score': [
                bleu_scores['bleu'],
                rouge_scores['rouge1']['f1'],
                rouge_scores['rouge2']['f1'],
                rouge_scores['rougeL']['f1'],
                bert_scores['f1'][0]
            ]
        }
        
        return pd.DataFrame(results)


# Initialize BERTScore evaluator
print("Initializing BERTScore evaluator...\n")
bertscore_evaluator = BERTScoreEvaluator('bert-base-uncased')

print("\n" + "="*80)
print("COMPARING BERTSCORE WITH TRADITIONAL METRICS")
print("="*80)

# Example where semantic meaning is preserved but words differ
reference = "The scientist conducted an experiment to test the hypothesis."

examples = [
    ("The researcher performed a test to verify the theory.",
     "Semantic equivalent with different words"),
    
    ("The scientist conducted an experiment to test the hypothesis.",
     "Perfect match"),
    
    ("The musician played a song at the concert.",
     "Completely different meaning"),
]

for candidate, description in examples:
    print(f"\n{description}:")
    print(f"  Reference: {reference}")
    print(f"  Candidate: {candidate}")
    print()
    
    comparison = bertscore_evaluator.compare_with_traditional(candidate, reference)
    print(comparison.to_string(index=False))
    print("\n" + "-"*80)

print("\n" + "="*80)
print("KEY OBSERVATIONS")
print("="*80)
print("""
Notice how BERTScore gives high scores to the semantically equivalent text
even though BLEU gives it a low score due to different word choices.

BERTScore Advantages:
  • Recognizes semantic equivalence across different wordings
  • Correlates better with human judgment than surface metrics
  • Leverages powerful pre-trained language understanding
  • Works across many languages (with appropriate models)

BERTScore Considerations:
  • More computationally expensive than traditional metrics
  • Requires loading a large pre-trained model
  • Inherits biases from the underlying embedding model
  • May not be suitable for very specific domains without fine-tuning

When to Use BERTScore:
  • Evaluating generation where semantic accuracy matters more than exact wording
  • Comparing paraphrases or translations
  • When you need better correlation with human judgment
  • For final evaluation (use faster metrics for development iteration)
""")

Initializing BERTScore evaluator...

BERTScore will use model: bert-base-uncased
Note: First run will download the model, which may take time.

COMPARING BERTSCORE WITH TRADITIONAL METRICS

Semantic equivalent with different words:
  Reference: The scientist conducted an experiment to test the hypothesis.
  Candidate: The researcher performed a test to verify the theory.



LookupError: 
**********************************************************************
  Resource [93mpunkt_tab[0m not found.
  Please use the NLTK Downloader to obtain the resource:

  [31m>>> import nltk
  >>> nltk.download('punkt_tab')
  [0m
  For more information see: https://www.nltk.org/data.html

  Attempted to load [93mtokenizers/punkt_tab/english/[0m

  Searched in:
    - '/Users/yenokhakobyan/nltk_data'
    - '/Users/yenokhakobyan/miniconda3/nltk_data'
    - '/Users/yenokhakobyan/miniconda3/share/nltk_data'
    - '/Users/yenokhakobyan/miniconda3/lib/nltk_data'
    - '/usr/share/nltk_data'
    - '/usr/local/share/nltk_data'
    - '/usr/lib/nltk_data'
    - '/usr/local/lib/nltk_data'
**********************************************************************


## Part 6: Task-Specific Evaluation - Question Answering

While general metrics like BLEU and ROUGE provide useful signals, evaluating performance on specific downstream tasks often gives more actionable insights. For question answering systems, we care about whether the model can correctly answer questions, not just whether it generates fluent text. This requires task-specific evaluation approaches that measure accuracy, F1 score for span extraction, or exact match rates.

Question answering evaluation typically comes in several forms. For extractive QA where answers are spans from a given passage, we measure exact match (the percentage of questions where the predicted answer exactly matches the gold answer) and F1 score (token-level overlap between predicted and gold answers, allowing partial credit). For open-domain QA where the model must retrieve information and generate answers, we might evaluate retrieval accuracy separately from answer accuracy.

For generative QA where models produce free-form answers rather than extracting spans, evaluation becomes more challenging. We might use reference-based metrics like ROUGE or BERTScore to compare generated answers to gold standards, or we might employ answer equivalence checking where we determine if two different phrasings convey the same factual information. Some modern approaches even use language models themselves as evaluators, having them judge whether answers are factually correct and complete.

In [ ]:
class QuestionAnsweringEvaluator:
    """Evaluate question answering systems with multiple metrics.
    
    This evaluator handles both extractive QA (where answers are text spans)
    and generative QA (where answers are generated freely). Different metrics
    apply to each type.
    """
    
    def __init__(self):
        """Initialize QA evaluator with necessary components."""
        pass
    
    def compute_exact_match(self, prediction: str, ground_truth: str) -> float:
        """Compute exact match score (binary).
        
        Returns 1.0 if the strings match exactly (after normalization),
        otherwise 0.0. This is a strict metric but commonly used for QA.
        """
        # Normalize strings: lowercase, strip whitespace, remove articles
        def normalize(s: str) -> str:
            s = s.lower().strip()
            # Remove articles
            for article in ['a', 'an', 'the']:
                s = s.replace(f' {article} ', ' ')
            # Remove extra whitespace
            s = ' '.join(s.split())
            return s
        
        return 1.0 if normalize(prediction) == normalize(ground_truth) else 0.0
    
    def compute_f1_score(self, prediction: str, ground_truth: str) -> float:
        """Compute token-level F1 score.
        
        This allows partial credit for answers that overlap with the ground truth
        even if they are not exact matches. More lenient than exact match.
        """
        # Tokenize
        pred_tokens = prediction.lower().split()
        truth_tokens = ground_truth.lower().split()
        
        # Compute overlap
        common = set(pred_tokens) & set(truth_tokens)
        
        if len(common) == 0:
            return 0.0
        
        precision = len(common) / len(pred_tokens)
        recall = len(common) / len(truth_tokens)
        
        f1 = 2 * (precision * recall) / (precision + recall)
        return f1
    
    def evaluate_qa_pairs(self, predictions: List[str], 
                         ground_truths: List[str]) -> Dict[str, float]:
        """Evaluate a list of QA pairs.
        
        Computes average exact match and F1 scores across all examples.
        """
        exact_matches = []
        f1_scores = []
        
        for pred, truth in zip(predictions, ground_truths):
            exact_matches.append(self.compute_exact_match(pred, truth))
            f1_scores.append(self.compute_f1_score(pred, truth))
        
        return {
            'exact_match': np.mean(exact_matches),
            'f1': np.mean(f1_scores),
            'num_examples': len(predictions)
        }
    
    def detailed_evaluation(self, predictions: List[str],
                           ground_truths: List[str],
                           questions: List[str]) -> pd.DataFrame:
        """Provide detailed per-example evaluation.
        
        This is useful for error analysis and understanding where the
        model succeeds or fails.
        """
        results = []
        
        for q, pred, truth in zip(questions, predictions, ground_truths):
            em = self.compute_exact_match(pred, truth)
            f1 = self.compute_f1_score(pred, truth)
            
            results.append({
                'question': q[:60] + '...' if len(q) > 60 else q,
                'prediction': pred[:40] + '...' if len(pred) > 40 else pred,
                'ground_truth': truth[:40] + '...' if len(truth) > 40 else truth,
                'exact_match': em,
                'f1_score': f1
            })
        
        return pd.DataFrame(results)


# Initialize QA evaluator
qa_evaluator = QuestionAnsweringEvaluator()

print("="*80)
print("QUESTION ANSWERING EVALUATION")
print("="*80)

# Sample QA examples
questions = [
    "What is the capital of France?",
    "When was the Declaration of Independence signed?",
    "Who wrote Romeo and Juliet?",
    "What is the speed of light?",
]

ground_truths = [
    "Paris",
    "1776",
    "William Shakespeare",
    "299,792,458 meters per second",
]

# Simulate different quality predictions
predictions_perfect = [
    "Paris",
    "1776",
    "William Shakespeare",
    "299,792,458 meters per second",
]

predictions_good = [
    "Paris",
    "July 4, 1776",  # Extra detail
    "Shakespeare",    # Missing first name
    "approximately 300 million meters per second",  # Close but not exact
]

predictions_poor = [
    "Lyon",           # Wrong city
    "1777",           # Wrong year
    "Charles Dickens", # Wrong author
    "fast",           # Too vague
]

# Evaluate each set
print("\nScenario 1: Perfect Predictions")
print("-" * 80)
results_perfect = qa_evaluator.evaluate_qa_pairs(predictions_perfect, ground_truths)
print(f"Exact Match: {results_perfect['exact_match']:.2%}")
print(f"F1 Score:    {results_perfect['f1']:.2%}")

print("\nScenario 2: Good but Imperfect Predictions")
print("-" * 80)
results_good = qa_evaluator.evaluate_qa_pairs(predictions_good, ground_truths)
print(f"Exact Match: {results_good['exact_match']:.2%}")
print(f"F1 Score:    {results_good['f1']:.2%}")
print("Note: F1 gives partial credit, EM does not")

print("\nScenario 3: Poor Predictions")
print("-" * 80)
results_poor = qa_evaluator.evaluate_qa_pairs(predictions_poor, ground_truths)
print(f"Exact Match: {results_poor['exact_match']:.2%}")
print(f"F1 Score:    {results_poor['f1']:.2%}")

# Detailed per-example analysis
print("\n" + "="*80)
print("DETAILED PER-EXAMPLE ANALYSIS (Good Predictions)")
print("="*80)
detailed = qa_evaluator.detailed_evaluation(predictions_good, ground_truths, questions)
print(detailed.to_string(index=False))

print("\n" + "="*80)
print("QA EVALUATION INSIGHTS")
print("="*80)
print("""
Exact Match vs F1:
  • Exact Match: Binary, strict metric. Answer must be exactly correct.
  • F1 Score: Allows partial credit for overlap. More lenient.
  • Both are useful: EM for strict evaluation, F1 for understanding quality

When to Use Each:
  • Exact Match: When precision is critical (e.g., medical diagnoses)
  • F1 Score: When partial answers have value (e.g., research assistance)
  • Both together: Provides complete picture of system performance

Common Challenges:
  • Multiple valid answers: "Paris" vs "the city of Paris"
  • Different formats: "1776" vs "July 4, 1776"
  • Synonyms: "Shakespeare" vs "William Shakespeare" vs "the Bard"
  • Approximations: "300 million m/s" vs exact value

Best Practices:
  • Use multiple reference answers when possible
  • Normalize strings (lowercase, remove articles, etc.)
  • Consider semantic equivalence checking for complex answers
  • Analyze errors qualitatively to understand failure modes
""")

## Conclusion: Building a Comprehensive Evaluation Strategy

We have now explored the full spectrum of evaluation approaches for language models, from automatic metrics through task-specific evaluation to human assessment and safety testing. The key insight is that no single metric or evaluation approach provides a complete picture of model quality. Instead, you need a comprehensive evaluation strategy that combines multiple perspectives.

Your evaluation approach should match your use case and constraints. For rapid development iteration, fast automatic metrics like perplexity or BLEU provide quick feedback. For validating model improvements, more sophisticated metrics like BERTScore or task-specific accuracy offer better signal. For final validation before deployment, human evaluation remains essential for catching issues that automatic metrics miss.

Remember that evaluation is not just about computing scores. It is about understanding your model's behavior, identifying failure modes, and making informed decisions about deployment. Use evaluation results to guide improvement efforts, catching problems early, and building trust with stakeholders who need to understand model capabilities and limitations.

As language models continue to advance, evaluation methods must evolve as well. Stay informed about new evaluation frameworks, participate in shared tasks and benchmarks, and always question whether your evaluation truly measures what you care about. The goal is not perfect scores on any particular metric, but rather building systems that work well and safely in their intended applications.
""")